# 02 — LoRA Fine-tuning (audio-grounded) + Push to HuggingFace

**Run on a 24 GB+ GPU notebook.**

Trains Moshi to emit `<|tool_call|>` in its text stream **when it hears** a
request — conditioned on real Mimi-encoded user audio (rows 9:17 of the code
tensor), not on text.

Steps:
1. Pull patched Moshi weights from `abrarfahim/moshi-tool-patched`
2. Load the audio dataset `abrarfahim/moshi-tool-audio` (codes/mask only)
3. Apply LoRA to the temporal-transformer attention (PEFT)
4. Fine-tune the **text stream** (`compute_loss`) conditioned on the user audio
5. Sanity check on real audio codes (positives fire, negatives stay silent)
6. Merge LoRA and push to `abrarfahim/moshi-tool-finetuned`

Config: batch 1 (variable-length audio) + grad-accum 16, LoRA rank 16, 6 epochs.

In [ ]:
import subprocess, torch
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                    "--format=csv,noheader"], capture_output=True, text=True)
print(r.stdout.strip())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")

In [ ]:
# Install into the EXACT python running this kernel (most reliable in Modal)
import sys
!{sys.executable} -m pip install peft==0.11.0 safetensors sentencepiece huggingface_hub einops sphn aiohttp numpy tqdm datasets -q
# verify they're importable in THIS kernel
import importlib
for m in ["peft", "safetensors", "sentencepiece", "einops", "sphn", "huggingface_hub", "tqdm", "datasets"]:
    importlib.import_module(m)
print("All deps importable ✓")

In [ ]:
import gc, json, os, shutil, subprocess, sys
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ── Clone repo ────────────────────────────────────────────────────────────────
REPO = Path("/repo")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone",
    "https://github.com/syedfahimabrar/moshi-D-gu.git",
    str(REPO)], check=True)
sys.path.insert(0, str(REPO / "moshi"))

import sentencepiece as spm
from moshi.models.loaders import get_moshi_lm
from huggingface_hub import hf_hub_download

# ── Config ────────────────────────────────────────────────────────────────────
HF_PATCHED_REPO = "abrarfahim/moshi-tool-patched"
HF_OUTPUT_REPO  = "abrarfahim/moshi-tool-finetuned"
HF_TOKEN        = os.environ.get("HF_TOKEN", None)  # or paste token here

LORA_RANK    = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05
GRAD_ACCUM   = 16    # batch size is fixed at 1 (variable-length audio) → accumulate
LR           = 2e-4
EPOCHS       = 6
MAX_SEQ_LEN  = 1024  # audio examples are longer than the old text-only ones

WEIGHTS_DIR = Path("/tmp/moshi_weights")
OUT_DIR     = Path("/mnt/model")
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE       = torch.bfloat16

PAD_ID             = 3
TOOL_CALL_ID       = 32000
TOOL_END_ID        = 32001
TOOL_RESULT_ID     = 32002
TOOL_RESULT_END_ID = 32003

OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE} | rank: {LORA_RANK} | grad_accum: {GRAD_ACCUM} | epochs: {EPOCHS}")

In [ ]:
from huggingface_hub import hf_hub_download

WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
for filename in ["model.safetensors", "tokenizer_spm_32k_3.model"]:
    dest = WEIGHTS_DIR / filename
    if dest.exists():
        print(f"Already present: {filename}")
        continue
    print(f"Downloading {filename} ...")
    hf_hub_download(repo_id=HF_PATCHED_REPO, filename=filename,
                    local_dir=str(WEIGHTS_DIR), token=HF_TOKEN)
    print(f"  ✓ {filename}")
print("Weights ready.")

In [ ]:
tok = spm.SentencePieceProcessor()
tok.Load(str(WEIGHTS_DIR / "tokenizer_spm_32k_3.model"))
assert tok.get_piece_size() == 32004, f"Expected 32004 tokens, got {tok.get_piece_size()}"
print(f"Tokenizer: {tok.get_piece_size()} tokens")

# Download the AUDIO-grounded dataset built by notebook 01.
# Drop the `audio` column immediately so it is never decoded (decoding the
# Audio feature would require torchcodec; training only needs codes/mask/type).
from datasets import load_dataset
HF_DATA_REPO = "abrarfahim/moshi-tool-audio"
ds = load_dataset(HF_DATA_REPO, split="train", token=HF_TOKEN)
keep = ["codes", "mask", "type"]
ds = ds.remove_columns([c for c in ds.column_names if c not in keep])
ds = ds.with_format("python")

from collections import Counter
print(f"Loaded {len(ds)} examples from {HF_DATA_REPO}")
print("Breakdown:", dict(Counter(ds["type"])))
print(f"codes e.g. {len(ds[0]['codes'])} x {len(ds[0]['codes'][0])}  "
      f"(17 rows = 1 text + 8 moshi + 8 user)")

In [ ]:
torch.cuda.empty_cache(); gc.collect()
free, total = torch.cuda.mem_get_info()
print(f"GPU: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

print("Loading model to CPU then moving to GPU ...")
lm_model = get_moshi_lm(WEIGHTS_DIR / "model.safetensors", device="cpu", dtype=DTYPE)
lm_model = lm_model.to(DEVICE)
lm_model.train()
print(f"Loaded: {sum(p.numel() for p in lm_model.parameters())/1e9:.2f}B params  text_card={lm_model.text_card}")
free2, _ = torch.cuda.mem_get_info()
print(f"GPU after load: {free2/1e9:.1f} GB free")

In [ ]:
from peft import LoraConfig, get_peft_model

for p in lm_model.parameters():
    p.requires_grad = False

lora_config = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=["out_proj", "in_proj"], bias="none",
)
lm_model = get_peft_model(lm_model, lora_config)
lm_model.print_trainable_parameters()

In [ ]:
class ToolAudioDataset(Dataset):
    def __init__(self, hf_ds, max_len=MAX_SEQ_LEN):
        self.examples = []
        for ex in hf_ds:
            codes = torch.tensor(ex["codes"], dtype=torch.long)[:, :max_len]  # [17,T]
            mask  = torch.tensor(ex["mask"],  dtype=torch.long)[:max_len]      # [T]
            self.examples.append({"codes": codes, "mask": mask})
        print(f"Dataset: {len(self.examples)} examples")

    def __len__(self): return len(self.examples)
    def __getitem__(self, i): return self.examples[i]

# Batch size 1 → no padding/collation needed (examples vary in length).
# We compensate with a larger GRAD_ACCUM for an effective batch.
def collate(batch):
    b = batch[0]
    return {"codes": b["codes"].unsqueeze(0),   # [1, 17, T]
            "mask":  b["mask"].unsqueeze(0)}     # [1, T]

dataset    = ToolAudioDataset(ds)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True,
                        collate_fn=collate, num_workers=0)
print(f"Batches/epoch: {len(dataloader)}  (effective batch = {GRAD_ACCUM})")

In [ ]:
_loss_weight = {"w": None}

def compute_loss(model, codes, mask):
    """codes: [B,17,T]. Trains the text row where mask=1, conditioned on the
    real user audio in rows 9:17. PAD/EPAD targets are downweighted so the many
    suppression frames don't drown out the call + reply tokens."""
    base = model.base_model.model if hasattr(model, 'base_model') else model
    out = base.forward_train(codes)
    tl = out.text_logits[:, 0].float()   # [B, T, card] -> float32 for stable loss
    vm = out.text_mask[:, 0]             # [B, T]
    flat_m = (mask.bool() & vm).reshape(-1)
    if flat_m.sum() == 0:
        return torch.tensor(0., requires_grad=True, device=codes.device)
    logits = tl.reshape(-1, tl.shape[-1])[flat_m]
    tgt    = codes[:, 0].reshape(-1)[flat_m]

    w = _loss_weight["w"]
    if w is None or w.numel() != tl.shape[-1]:
        w = torch.ones(tl.shape[-1], device=codes.device, dtype=torch.float32)
        w[PAD_ID] = 0.3   # PAD (3): keep suppression but don't let it dominate
        w[0]      = 0.3   # EPAD (0)
        _loss_weight["w"] = w
    return nn.functional.cross_entropy(logits, tgt, weight=w)

In [ ]:
from tqdm.auto import tqdm

optimizer = torch.optim.AdamW(
    [p for p in lm_model.parameters() if p.requires_grad],
    lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS * len(dataloader))

lm_model.train()
global_step = 0
optimizer.zero_grad()

for epoch in range(EPOCHS):
    epoch_loss, n = 0., 0
    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=True)
    for step, batch in enumerate(pbar):
        codes = batch["codes"].to(DEVICE)
        mask  = batch["mask"].to(DEVICE)
        loss  = compute_loss(lm_model, codes, mask) / GRAD_ACCUM
        loss.backward()
        epoch_loss += loss.item() * GRAD_ACCUM
        n += 1
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(
                [p for p in lm_model.parameters() if p.requires_grad], 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            global_step += 1
        pbar.set_postfix(loss=f"{epoch_loss/n:.4f}",
                         lr=f"{scheduler.get_last_lr()[0]:.2e}")
    print(f"=== Epoch {epoch+1} done  loss={epoch_loss/n:.4f} ===")

print("Training complete.")

In [ ]:
from safetensors.torch import save_file

print("Merging LoRA weights ...")
merged = lm_model.merge_and_unload()
merged.eval()

raw_sd = merged.state_dict()
PREFIX = "base_model.model."
while any(k.startswith(PREFIX) for k in raw_sd):
    raw_sd = {(k[len(PREFIX):] if k.startswith(PREFIX) else k): v for k, v in raw_sd.items()}

out_weight = OUT_DIR / "model.safetensors"
save_file({k: v.contiguous() for k, v in raw_sd.items()}, str(out_weight))
shutil.copy2(WEIGHTS_DIR / "tokenizer_spm_32k_3.model",
             OUT_DIR / "tokenizer_spm_32k_3.model")
print(f"Saved → {OUT_DIR}")

# raw_sd holds references to every GPU weight tensor — drop it so the GPU can be
# freed for the sanity check (otherwise it pins ~14 GB).
del raw_sd
import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Sanity check on GPU — reuse the already-trained `merged` model (no second copy).
import gc

if "merged" in globals():
    check = merged.base_model.model if hasattr(merged, "base_model") else merged
else:
    # rerun path: properly free the GPU, then reload the saved model on GPU.
    # raw_sd (merged.state_dict()) pins every weight tensor — must drop it too.
    for _n in ["lm_model", "optimizer", "scheduler", "dataloader",
               "loss", "batch", "out", "raw_sd", "check"]:
        globals().pop(_n, None)
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU freed → {free/1e9:.1f}/{total/1e9:.1f} GB free")
    check = get_moshi_lm(OUT_DIR / "model.safetensors", device=DEVICE, dtype=torch.bfloat16)

check.eval()

@torch.no_grad()
def predict_text(codes):
    out = check.forward_train(codes.long().unsqueeze(0).to(DEVICE))
    return out.text_logits[0, 0].argmax(-1).cpu()   # [T] predicted text per frame

# Group examples per type
by_type = {}
for ex, typ in zip(dataset.examples, ds["type"]):
    by_type.setdefault(typ, []).append(ex)

POS = ["time", "weather", "weather_local", "time_multiturn", "weather_multiturn"]
NEG = ["chitchat", "distractor", "silence"]

print("── Positives: model should predict <|tool_call|> where trained ──")
pos_ok = pos_tot = 0
for t in POS:
    d = by_type[t][0]
    pred = predict_text(d["codes"])
    tgt  = d["codes"][0]
    call_pos = (tgt == TOOL_CALL_ID).nonzero().flatten().tolist()
    hit = len(call_pos) > 0 and all(pred[p].item() == TOOL_CALL_ID for p in call_pos)
    pos_ok += hit; pos_tot += 1
    print(f"  {'✓' if hit else '✗'} {t:20s} call positions={call_pos} "
          f"predicted={[pred[p].item() for p in call_pos]}")

print("\n── Negatives: model should NOT predict <|tool_call|> anywhere ──")
neg_ok = neg_tot = 0
for t in NEG:
    clean_all = True
    for d in by_type[t][:5]:
        pred = predict_text(d["codes"])
        m = d["mask"].bool()
        if (pred[m] == TOOL_CALL_ID).any():
            clean_all = False
    neg_ok += clean_all; neg_tot += 1
    print(f"  {'✓' if clean_all else '✗'} {t:20s} (no tool call across 5 samples)")

print(f"\nPositives: {pos_ok}/{pos_tot}   Negatives clean: {neg_ok}/{neg_tot}")
if pos_ok == pos_tot and neg_ok == neg_tot:
    print("✓ Audio-grounded tool calling learned!")
else:
    print("⚠ Mixed — consider more epochs or more data for the weak category.")

In [ ]:
from huggingface_hub import HfApi, create_repo

api = HfApi(token=HF_TOKEN)
create_repo(HF_OUTPUT_REPO, private=True, exist_ok=True, token=HF_TOKEN)
print(f"Repo: https://huggingface.co/{HF_OUTPUT_REPO}")

for f in OUT_DIR.iterdir():
    if not f.is_file():
        continue
    print(f"Uploading {f.name}  ({f.stat().st_size/1e9:.2f} GB) ...")
    api.upload_file(path_or_fileobj=str(f), path_in_repo=f.name,
                    repo_id=HF_OUTPUT_REPO, token=HF_TOKEN)
    print(f"  ✓ {f.name}")

print(f'''
Upload complete!

On your server:
  bash run_server.sh --hf-model {HF_OUTPUT_REPO}
''')